# Sample run for Fisher Market

In [1]:
using Pkg
Pkg.activate("../")

  Activating project at `~/workspace/ExchangeMarket.jl/scripts`


In [2]:
using Revise
using Random, SparseArrays, LinearAlgebra
using JuMP, MosekTools
using Plots, LaTeXStrings, Printf
import MathOptInterface as MOI

using ExchangeMarket

include("../tools.jl")
include("../plots.jl")
include("./setup.jl")
switch_to_pdf(; bool_use_html=false)

:pdf

# Solving using approximate UMP

In [3]:
Random.seed!(1)
n = 500
m = 1000
ρ = 1.0
# method_filter(name) = name ∈ [:LogBar]
# method_filter(name) = name ∈ [:Tât, :PropRes]
method_filter(name) = name ∈ [:PropRes]

method_filter (generic function with 1 method)

In [4]:
table_time = []
results = []
results_phi = Dict()
results_ground = Dict()

f0 = FisherMarket(m, n; ρ=1.0, bool_unit=true, scale=30.0,
    sparsity=0.05,
    ε_br_play=1e-1
)
linconstr = LinearConstr(1, n, ones(1, n), [sum(f0.w)])

# -----------------------------------------------------------------------
# compute ground truth
# -----------------------------------------------------------------------
# μ₀ = 1e1
f1 = copy(f0)
p₀ = ones(n) * sum(f1.w) ./ (n)
# p₀ = ones(n) .* μ₀
x₀ = ones(n, m) ./ m
f1.x = copy(x₀)
f1.p .= p₀;

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0796 seconds
FisherMarket initialized in 0.1326 seconds
FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0009 seconds
FisherMarket initialized in 0.0047 seconds


In [ ]:
# use proportional response method to compute ground truth
(name, method, kwargs) = method_kwargs[end]
alg = method(
    n, m, copy(p₀);
    linconstr=linconstr,
    kwargs...
)
traj = opt!(
    alg, f1;
    keep_traj=true,
    bool_init_phase=true,
    maxiter=2000,
    tol=1e-3
)
results_ground[ρ] = (alg, traj, f1)
results_phi[ρ] = copy(alg.p);
push!(results, ((name, ρ), (alg, traj[2:end], f1)))
push!(table_time, (n, m, name, ρ, traj[end].t))
validate(f1, alg)

## ApproxLin

In [ ]:
# self-checking.
bool_run_approxlin = true
if bool_run_approxlin
    (name, method, kwargs) = [
        :LogBar,
        HessianBar,
        Dict(
            :tol => 1e-12, :maxiter => 20,
            :optimizer => ApproxLin,
            # :optimizer => CESConic,
            # :optimizer => DualLPConic,
            # :option_mu => :pred_corr,
            :option_mu => :normal,
            # :option_mu => :linear,
            :option_step => :affinesc,
            # :option_step => :homotopy,
            # :option_step => :logbar,
            # :linsys => :krylov,
            # :linsys => :DRq,
            :linsys => :direct,
        )
    ]
    f1 = copy(f0)
    p₀ = ones(n) * sum(f1.w) ./ (n)
    x₀ = ones(n, m) ./ m
    f1.x = copy(x₀)
    f1.p .= p₀
    alg = method(
        n, m, p₀;
        linconstr=linconstr,
        kwargs...
    )
    f1.ε_br_play .= 1e-1
    alg.μₗ = 1e-10
    traj = opt!(
        alg, f1;
        maxiter=200,
        pₛ=results_phi[ρ],
        keep_traj=true
    )
    push!(results, ((name, ρ), (alg, traj[2:end], f1)))
    push!(table_time, (n, m, name, ρ, traj[end].t))
end

## LSE (logsumexp-smoothed) dual solver

In [ ]:
min(10 / (log(n) + 6))

In [ ]:
# self-checking.
(name, method, kwargs) = [
    :LSE,
    HessianBar,
    Dict(
        :tol => 1e-12, :maxiter => 20,
        :optimizer => LSEResponse,
        # :option_mu => :normal,
        # :option_mu => :pred_corr,
        :option_mu => :nothing,
        # :option_step => :affinesc,
        :option_step => :logbar,
        # :option_step => :homotopy,
        :linsys => :direct,
        # :linsys => :krylov,
    )
]
f1 = copy(f0)
p₀ = ones(n) * sum(f1.w)
x₀ = ones(n, m) ./ m
f1.x = copy(x₀)
f1.p .= p₀
alg = method(
    n, m, p₀;
    linconstr=linconstr,
    kwargs...
)
f1.ε_br_play .= min(10 / (log(n) + 6), 6e-1)
alg.μₗ = 1e-20
traj = opt!(
    alg, f1;
    maxiter=20,
    pₛ=results_phi[ρ],
    keep_traj=true
)
push!(results, ((name, ρ), (alg, traj[2:end], f1)))
push!(table_time, (n, m, name, ρ, traj[end].t))

In [ ]:
(1.1 + √n) / (3.2 + √n)

In [ ]:
sum(alg.p)

In [ ]:
alg.y

## Validate the ground truth

In [ ]:
# for (ρ, (alg, traj, f1)) in results
for (ρ, (alg, traj, f1)) in results
    validate(f1, alg)
end

In [ ]:
[results[1][2][1].p results[2][2][1].p]

## Plot trajectory 

for each $\rho$, we plot the distance to ground truth $\|p - p^*\|$ of the trajectory.

In [ ]:
ffs = []

for attr in [:k]
    ρfmt = @sprintf("%+.2f", ρ)
    σfmt = @sprintf("%+.2f", ρ / (1 - ρ))
    fig = generate_empty(; shape=:wide)
    plot!(
        fig,
        ylabel=L"$\|\mathbf{p} - \mathbf{p}^*\|$",
        title=L"$\rho := %$ρfmt~(\sigma := %$σfmt)$",
        legendbackgroundcolor=RGBA(1.0, 1.0, 1.0, 0.8),
        yticks=10.0 .^ (-16:4:3),
        xtickfont=font(18),
        ytickfont=font(18),
        xscale=:identity,
        size=(1200, 600)
    )
    if attr == :k
        plot!(
            fig,
            xticks=[10, 20, 50, 100, 200, 500]
        )
    end
    for ((mm, _ρ), (alg, traj, f1)) in results
        if _ρ != ρ
            continue
        end
        traj_pp₊ = map(pp -> pp.D, traj)
        traj_tt₊ = map(pp -> getfield(pp, attr), traj)
        println(traj_pp₊)
        plot!(fig, traj_tt₊, traj_pp₊ .+ 1e-20, label=L"\texttt{%$mm}", linewidth=2, linestyle=:dash, markershape=:circle)
    end
    push!(ffs, fig)
end

In [ ]:
ff = plot(ffs...)

In [ ]:
savefig(ff, "/tmp/test_ipm_linear.pdf")

# Primal-Dual IPM

In [5]:
f0.sparsity

0.05

In [6]:
include("./linear-pd.jl")

fpd = copy(f0)
(; st, traj, alg) = pd_ipm(
    fpd;
    μ₀=1.0, maxiter=100,
    mode=:exact, tol=1e-9, linalg=:auto
)

pd_allocation!(fpd, st, alg)
validate(fpd, alg)

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0010 seconds
FisherMarket initialized in 0.0140 seconds
-------------------------------------------------------------------------------------------------------------
                                     ExchangeMarket.jl: Primal-Dual IPM
                                  mode=:exact, linalg=dense (n=500, m=1000)
-------------------------------------------------------------------------------------------------------------
    k |          μ |       |mc| |       |r₂| |       |r₃| |       |c₁| |       |c₂| |          α |    time(s)
-------------------------------------------------------------------------------------------------------------
    1 | +1.000e+00 |  5.152e+05 |  5.242e+05 |  4.337e-19 |  0.000e+00 |  1.110e-16 |     0.9971 |     0.2298
      |- t_residuals=0.0150  t_update_kkt=0.0026  t_schur=0.2058  t_backsub=0.0031
    2 | +2.778e-05 |  2.471e+03 |  1.490e+03 |  1.735e-18 |  9.884e-03 |  9.884e-0

In [ ]:
alg.kkt.S_sparse

In [ ]:
sparse(alg.kkt.Wc)